# CryptoTrade Pro — AI/ML Security Audit

This notebook walks through a full AI/ML security audit of the **CryptoTrade Pro** trading platform.

You will learn how to:
- Identify ML-specific vulnerabilities
- Detect unsafe deserialization (pickle)
- Detect hardcoded secrets
- Detect command injection in data pipelines
- Scan notebooks for insecure patterns
- Use Bandit, Semgrep, Safety, and Pylint
- Generate a professional audit summary

This notebook mirrors the workflow used in the **SEC-mandated 72-hour breach audit** described in the course.

## 1. Environment Setup

In [1]:
!pip install bandit semgrep safety pylint pandas tabulate -q
import os
os.environ['PATH'] += f":{os.path.expanduser('~')}/.local/bin"

# Ensure output directories exist
os.makedirs('security-reports', exist_ok=True)
os.makedirs('src', exist_ok=True)

print('Environment ready.')

Environment ready.


## 2. Project Structure Overview

The CryptoTrade Pro project contains:

- `src/model_management.py` — vulnerable model loading
- `src/data_pipeline.py` — vulnerable data pipeline
- `rules/*.yaml` — custom Semgrep rules
- `notebooks/data_pipeline.ipynb` — vulnerable notebook
- `trades/data.csv` — sample trading data

We will scan all of these.

## 3. Bandit — Python Static Analysis

In [25]:
!ls ../src/

data_pipeline.ipynb  model_management.py	trading_pipeline.py
model_loader.py      model_management_fixed.py


In [24]:
!pwd


/workspaces/Secure-AI-Code-and-Libraries-/Module 2 AI-Specific Code Vulnerabilities/Hands-On-Learning: Financial ML Model Security Audit/Notebooks


In [30]:
!bandit -r ../src/ -f json -o security-reports/bandit-results.json || true
print("Bandit report saved.")


[main]	INFO	profile include tests: None
[main]	INFO	profile exclude tests: None
[main]	INFO	cli include tests: None
[main]	INFO	cli exclude tests: None
[json]	INFO	JSON output written to file: security-reports/bandit-results.json
Bandit report saved.


In [31]:
!ls security-reports

bandit-results.json  bandit.json  semgrep.json


In [32]:
import json

with open('security-reports/bandit-results.json') as f:
    bandit = json.load(f)

print(f"Total Bandit findings: {len(bandit.get('results', []))}")

Total Bandit findings: 12


In [33]:
bandit_findings = [
    {
        'severity': r.get('issue_severity'),
        'rule': r.get('test_id'),
        'cwe': r.get('issue_cwe', {}).get('id') if isinstance(r.get('issue_cwe'), dict) else r.get('issue_cwe'),
        'file': r.get('filename'),
        'line': r.get('line_number'),
        'text': r.get('issue_text')
    }
    for r in bandit.get('results', [])
]
bandit_findings[:10]

[{'severity': 'LOW',
  'rule': 'B403',
  'cwe': 502,
  'file': '../src/model_loader.py',
  'line': 4,
  'text': 'Consider possible security implications associated with pickle module.'},
 {'severity': 'MEDIUM',
  'rule': 'B301',
  'cwe': 502,
  'file': '../src/model_loader.py',
  'line': 19,
  'text': 'Pickle and modules that wrap it can be unsafe when used to deserialize untrusted data, possible security issue.'},
 {'severity': 'LOW',
  'rule': 'B403',
  'cwe': 502,
  'file': '../src/model_management.py',
  'line': 1,
  'text': 'Consider possible security implications associated with pickle module.'},
 {'severity': 'LOW',
  'rule': 'B105',
  'cwe': 259,
  'file': '../src/model_management.py',
  'line': 8,
  'text': "Possible hardcoded password: 'SuperSecret99'"},
 {'severity': 'LOW',
  'rule': 'B105',
  'cwe': 259,
  'file': '../src/model_management.py',
  'line': 11,
  'text': "Possible hardcoded password: 'NhqPtmdSJYdKjVHjA7PZj4Mge3R5YNiP1e3UZjInClVN65y2H'"},
 {'severity': 'LOW',
  

### Bandit Severity, CWE, and File Breakdown

In [34]:
import pandas as pd

df_bandit = pd.DataFrame(bandit_findings)

if df_bandit.empty:
    print('No Bandit findings.')
else:
    print('=== Bandit Severity Breakdown ===')
    print(df_bandit['severity'].value_counts(), '\n')

    print('=== Bandit Findings by CWE ===')
    print(df_bandit.groupby('cwe').size(), '\n')

    print('=== Bandit Findings by File ===')
    print(df_bandit.groupby('file').size(), '\n')

=== Bandit Severity Breakdown ===
severity
LOW       6
MEDIUM    6
Name: count, dtype: int64 

=== Bandit Findings by CWE ===
cwe
259    4
400    1
502    7
dtype: int64 

=== Bandit Findings by File ===
file
../src/model_loader.py         2
../src/model_management.py    10
dtype: int64 



## 4. Semgrep — Custom ML Security Rules

In [46]:
# Semgrep exits with code 1 when it finds issues — '|| true' prevents cell failure
!semgrep --config ../rules ../src/ --json --output security-reports/semgrep.json || true
print("Semgrep report saved.")





┌──── ○○○ ────┐
│ Semgrep CLI │
└─────────────┘

⠧ Loading rules...                                                                                
Scanning 5 files (only git-tracked) with 10 Code rules:
            
  CODE RULES
  Scanning 4 files with 10 python rules.
                    
  SUPPLY CHAIN RULES
                  
  No rules to run.
                  
          
  PROGRESS
   
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100% 0:00:00                                                                                ━━━━━━━━━━━━━━━━━━━━━━━━   0% -:--:--
                
                
┌──────────────┐
│ Scan Summary │
└──────────────┘
✅ Scan completed successfully.
 • Findings: 19 (19 blocking)
 • Rules run: 10
 • Targets scanned: 4
 • Parsed lines: ~100.0%
 • Scan was limited to files tracked by git
 • For a detailed list of skipped files and lines, run semgrep with the --verbose flag
Ran 10 rules on 4 files: 19 findings.
Semgrep report saved.


In [47]:
with open('security-reports/semgrep.json') as f:
    semgrep = json.load(f)

print(f"Total Semgrep findings: {len(semgrep.get('results', []))}")

Total Semgrep findings: 19


In [48]:
semgrep_findings = [
    {
        'rule': r.get('check_id'),
        'file': r.get('path'),
        'line': r.get('start', {}).get('line'),
        'message': r.get('extra', {}).get('message')
    }
    for r in semgrep.get('results', [])
]
semgrep_findings[:10]

[{'rule': 'rules.unsafe-keras-load-model',
  'file': '../src/model_loader.py',
  'line': 11,
  'message': 'tf.keras.models.load_model() without integrity checks — model files can be tampered with.'},
 {'rule': 'rules.unsafe-pickle-load',
  'file': '../src/model_loader.py',
  'line': 19,
  'message': 'pickle.load() without validation — insecure deserialization can lead to remote code execution.'},
 {'rule': 'rules.hardcoded-api-key',
  'file': '../src/model_management.py',
  'line': 8,
  'message': 'Hardcoded API key detected — secrets must not be stored in source code.'},
 {'rule': 'rules.hardcoded-api-key',
  'file': '../src/model_management.py',
  'line': 9,
  'message': 'Hardcoded API key detected — secrets must not be stored in source code.'},
 {'rule': 'rules.hardcoded-api-key',
  'file': '../src/model_management.py',
  'line': 10,
  'message': 'Hardcoded API key detected — secrets must not be stored in source code.'},
 {'rule': 'rules.hardcoded-api-key',
  'file': '../src/model_m

### Semgrep Rule and File Breakdown

In [49]:
df_semgrep = pd.DataFrame(semgrep_findings)

if df_semgrep.empty:
    print('No Semgrep findings.')
else:
    print('=== Semgrep Findings by Rule ===')
    print(df_semgrep.groupby('rule').size(), '\n')

    print('=== Semgrep Findings by File ===')
    print(df_semgrep.groupby('file').size(), '\n')

=== Semgrep Findings by Rule ===
rule
rules.hardcoded-api-key          10
rules.unsafe-joblib-load          2
rules.unsafe-keras-load-model     2
rules.unsafe-pickle-load          4
rules.unsafe-pickle-loads         1
dtype: int64 

=== Semgrep Findings by File ===
file
../src/model_loader.py               2
../src/model_management.py          15
../src/model_management_fixed.py     2
dtype: int64 



## 5. Combined Vulnerability Table

In [61]:
import pandas as pd

combined = []

# -------------------------
# Bandit findings
# -------------------------
for r in bandit_findings:
    combined.append({
        'tool': 'Bandit',
        'severity': r.get('severity'),
        'rule': r.get('rule'),
        'cwe': r.get('cwe'),
        'file': r.get('file'),
        'line': r.get('line'),
        'message': r.get('text')
    })

# -------------------------
# Semgrep findings
# -------------------------
for r in semgrep_findings:
    combined.append({
        'tool': 'Semgrep',
        'severity': 'N/A',
        'rule': r.get('rule'),
        'cwe': 'N/A',
        'file': r.get('file'),
        'line': r.get('line'),
        'message': r.get('message')
    })

# -------------------------
# Build DataFrame
# -------------------------
df_combined = pd.DataFrame(combined)

# Add Finding_ID (F-01, F-02, ...)
df_combined['Finding_ID'] = [
    f"F-{i:02d}" for i in range(1, len(df_combined) + 1)
]

# Reorder columns
df_combined = df_combined[
    ['Finding_ID', 'tool', 'severity', 'rule', 'cwe', 'file', 'line', 'message']
]

# Display
from IPython.display import display
display(df_combined.head(20))

print("Combined shape:", df_combined.shape)


,Finding_ID,tool,severity,rule,cwe,file,line,message
0,F-01,Bandit,LOW,B403,502,../src/model_loader.py,4,Consider possible security implications associ...
1,F-02,Bandit,MEDIUM,B301,502,../src/model_loader.py,19,Pickle and modules that wrap it can be unsafe ...
2,F-03,Bandit,LOW,B403,502,../src/model_management.py,1,Consider possible security implications associ...
3,F-04,Bandit,LOW,B105,259,../src/model_management.py,8,Possible hardcoded password: 'SuperSecret99'
4,F-05,Bandit,LOW,B105,259,../src/model_management.py,11,Possible hardcoded password: 'NhqPtmdSJYdKjVHj...
5,F-06,Bandit,LOW,B105,259,../src/model_management.py,12,Possible hardcoded password: 'Bearer prod_abc1...
6,F-07,Bandit,LOW,B105,259,../src/model_management.py,14,Possible hardcoded password: 'wJalrXUtnFEMI/K7...
7,F-08,Bandit,MEDIUM,B301,502,../src/model_management.py,24,Pickle and modules that wrap it can be unsafe ...
8,F-09,Bandit,MEDIUM,B301,502,../src/model_management.py,35,Pickle and modules that wrap it can be unsafe ...
9,F-10,Bandit,MEDIUM,B113,400,../src/model_management.py,43,Call to requests without timeout


Combined shape: (31, 8)


### Auto-Number Findings (F-01 → F-XX)

In [62]:
if not df_combined.empty:
    df_combined = df_combined.sort_values(by=['file', 'line']).reset_index(drop=True)
    df_combined['Finding_ID'] = ['F-' + str(i+1).zfill(2) for i in range(len(df_combined))]
    df_combined[['Finding_ID','severity','rule','cwe','file','line','message']].head(20)
else:
    print('No findings to number.')

## 6. Final Audit Summary

In [63]:
print('=== FINAL AUDIT SUMMARY ===')
print(f"Total Findings: {len(df_combined)}")
print(f"Bandit Findings: {len(bandit_findings)}")
print(f"Semgrep Findings: {len(semgrep_findings)}")

if not df_bandit.empty:
    print('\nSeverity Breakdown:')
    print(df_bandit['severity'].value_counts())

if not df_combined.empty:
    print('\nTop Vulnerability Types:')
    print(df_combined['rule'].value_counts().head(10))

=== FINAL AUDIT SUMMARY ===
Total Findings: 31
Bandit Findings: 12
Semgrep Findings: 19

Severity Breakdown:
severity
LOW       6
MEDIUM    6
Name: count, dtype: int64

Top Vulnerability Types:
rule
rules.hardcoded-api-key          10
B301                              5
rules.unsafe-pickle-load          4
B105                              4
B403                              2
rules.unsafe-keras-load-model     2
rules.unsafe-joblib-load          2
B113                              1
rules.unsafe-pickle-loads         1
Name: count, dtype: int64


## 7. Export Markdown Report

In [64]:
if not df_combined.empty:
    with open('security-reports/audit_summary.md', 'w') as f:
        f.write(df_combined.to_markdown(index=False))
    print('Markdown report saved to security-reports/audit_summary.md')
else:
    print('No findings — skipping markdown export.')

Markdown report saved to security-reports/audit_summary.md
